In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

import pandas as pd
import utils.parser as parser
from sklearn.metrics import accuracy_score, recall_score, precision_score

import os
import random

In [1]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/home/robcli/.conda/envs/project/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/robcli/.conda/envs/project/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/robcli/.conda/envs/project/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/robcli/.conda/envs/project/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/robcli/.conda/envs/project/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/robcli/.conda/envs/project/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/robcli/.conda/envs/project/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in l

In [2]:
class MicrobiomeDataset(Dataset): 
    def __init__(self, fasta_files, labels, k: int=6, size: int=None, trim_to: int=None):
        self.fasta_files = fasta_files
        self.labels = labels
        self.label_tokenizer = {"nonIBD":0, "CD":1, "UC": 2}
        self.k = k
        self.vocab = parser.build_vocab(k=k)
        self.sequences = []
        
        for i, file in enumerate(fasta_files):
            if size is not None and i >= size:
                break

            data = parser.read_tokenized_file(file)
            if trim_to is not None:
                data = random.sample(data, k=trim_to)
                
            self.sequences.append((data, 
                                  self.label_tokenizer[labels[i]]))
            
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        tokenized_sequences, label = self.sequences[idx]
        return torch.tensor(tokenized_sequences), torch.tensor(label)

In [3]:
metadata = pd.read_csv("SJBae/sampled_metadata_contig_ACBI_2x.csv")

sampled_metadata = pd.concat([
           metadata[metadata["diagnosis"] == "UC"][:10], 
           metadata[metadata["diagnosis"] == "CD"][:10], 
           metadata[metadata["diagnosis"] == "nonIBD"][:10]]
         ).reset_index()

In [4]:
tokenized_files_dir = "/pool001/robcli/hmp2_tokenized"
tokenized_files = [os.path.join(tokenized_files_dir, f+".pkl") for f in sampled_metadata["External ID"]]
labels = sampled_metadata["diagnosis"]

In [5]:
class MultiheadAttentionPooling(nn.Module):
    """
    Pooling mechanism that uses multihead attention to create a weighted average of sequence embeddings.
    """
    def __init__(self, d_model: int=768, num_heads: int=12):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        
        # Query vector to attend to the sequence
        self.query = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Multi-head attention for pooling
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, microbiome_embeddings: torch.Tensor):
        """
        Apply attention pooling to a sequence of embeddings.
        Args:
            microbiome_embeddings: Tensor of shape (variable_length, d_model)  
        Returns:
            Tensor of shape (batch, d_model) containing pooled representation
        """
        microbiome_embeddings = microbiome_embeddings.unsqueeze(0)  # (1, variable_length, d_model)
        batch_size, var_len, _ = microbiome_embeddings.shape
        
        # Expand query to match batch size
        query = self.query.expand(batch_size, -1, -1)  # (batch, 1, d_model)
        
        # Apply multi-head attention to get weighted sum
        pooled_output, _ = self.multihead_attn(
            query=query,                      # (batch, 1, d_model)
            key=microbiome_embeddings,          # (batch, variable_length, d_model)
            value=microbiome_embeddings,        # (batch, variable_length, d_model)
        )
        
        # Remove the 2nd dimension (batch, 1, d_model)
        pooled_output = pooled_output.squeeze(1)  # (batch, d_model)
        return pooled_output

class MeanPooling(nn.Module):
    """
    Pooling mechanism that uses mean pooling to create an average of sequence embeddings.
    """
    def __init__(self):
        super().__init__()

    def forward(self, microbiome_embeddings: torch.Tensor, padding_token: int=0):
        """
        Apply mean pooling to a sequence of embeddings.
        Args:
            microbiome_embeddings: Tensor of shape (variable_length, d_model)  
        Returns:
            Tensor of shape (batch, d_model) containing mean pooled representation
        """
        masked = (microbiome_embeddings == padding_token).float()  # Shape: (variable_length, d_model)
        summed_embeddings = torch.sum(microbiome_embeddings, dim=0)
        token_count = torch.sum(masked, dim=0).clamp(min=1.0)
        pooled_output = summed_embeddings / token_count  # Shape: (d_model)
 
        return pooled_output.unsqueeze(0)  # (batch, d_model)

class MicrobiomeAttentionPooler(nn.Module):
    """
    Module that applies attention pooling to each microbiome.
    """
    def __init__(self, pooling: str="mean", d_model: int=768, n_heads: int=12):
        super().__init__()
        self.d_model = d_model

        if pooling == "mean":
            self.microbiome_pooler = MeanPooling()
        else:
            self.microbiome_pooler = MultiheadAttentionPooling(d_model=d_model, n_heads=n_heads)
        self.projection = nn.Linear(d_model, d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, batch_embeddings):
        """
        Pool a batch of microbiome embeddings at the microbiome levels.        
        Args:
            batch_embeddings: List of lists of sequence embeddings
                Each sequence embedding is a tensor of shape (d_model)
        Returns:
            List of microbiome-level embeddings, each of shape (d_model)
        """
        batch_pooled = []
        
        for microbiome_embeddings in batch_embeddings:
            # microbiome_embeddings have shape: (variable_length, d_model)
            
            # Pool all tokens directly to get microbiome representation
            microbiome_pooled_embedding = self.microbiome_pooler(microbiome_embeddings)  # (d_model)
            
            # Apply projection and normalization
            microbiome_pooled_embedding = self.projection(microbiome_pooled_embedding)
            microbiome_pooled_embedding = self.layer_norm(microbiome_pooled_embedding)
            
            batch_pooled.append(microbiome_pooled_embedding)
        
        return torch.cat(batch_pooled)  # Shape: (batch_size, d_model)

In [6]:
class MicrobiomeLM(nn.Module):
    def __init__(self, k: int=6, seq_len: int=512, num_outputs: int=3, d_model: int=768, n_heads: int=12, n_layers: int=12, dropout: float=0.1, pooling: str="mean"):
        super().__init__()

        self.seq_len = seq_len + 2 # Including start and stop tokens
        vocab_size = 4**k + 3 # Including start, pad, and stop tokens
        
        self.positional_embedder = nn.Embedding(self.seq_len, d_model)
        self.token_embedder = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.pooler = MicrobiomeAttentionPooler(pooling=pooling, d_model=d_model, n_heads=n_heads)
        self.FC = nn.Linear(d_model, d_model) 
        self.activation_fn = nn.ReLU()
        self.output_layer = nn.Linear(d_model, num_outputs)
        
    def forward(self, batch_sequences):
        batch_embeddings = [] # Shape: (batch, variable_length, seq_len, d_model)

        batch_size = batch_sequences.shape[0]
        for i in range(batch_size):
            microbiome = batch_sequences[i]  # Shape: (variable_length, seq_len)
            sequence_embeddings = self._process_microbiome(microbiome)
            batch_embeddings.append(sequence_embeddings) 

        x = self.pooler(batch_embeddings)
        x = self.FC(x)
        x = self.activation_fn(x)
        logits = self.output_layer(x)
        return logits

    def _process_microbiome(self, microbiome):
        """
        Process a single microbiome containing multiple sequences and return list of embeddings per sequence.
        """
        sequence_embeddings = []
        
        for seq_idx in range(microbiome.shape[0]):
            sequence = microbiome[seq_idx]
            sequence_embedding = self._encode_sequence(sequence)
            sequence_embeddings.append(sequence_embedding)
                    
        return torch.stack(sequence_embeddings) # Shape: (variable_length, d_model)

    def _encode_sequence(self, sequence, padding_token: int=0):
        """
        Encode a sequence of tokenized kmers by the encoder block. 
        """
        sequence = sequence.unsqueeze(0)  # Add batch dimension

        padding_mask = (sequence != padding_token).float()  # Shape (1, seq_len)
        
        sequence_embedding = self.token_embedder(sequence)  # Shape: (1, seq_len, d_model)
        positions = torch.arange(sequence.shape[1], device=sequence.device).unsqueeze(0)
        position_embedding = self.positional_embedder(positions)  # Shape: (1, seq_len, d_model)
        
        embedding = sequence_embedding + position_embedding
        attended_embedding = self.transformer(embedding)  # Shape: (1, seq_len, d_model)

        padding_mask = padding_mask.unsqueeze(-1).expand_as(attended_embedding)  # Shape (1, seq_len, d_model)
        masked_embedding = padding_mask * attended_embedding

        summed_embedding = torch.sum(masked_embedding, dim=1)
        token_count = torch.sum(padding_mask, dim=1).clamp(min=1.0)  # Prevent zero division
        mean_embedding = summed_embedding / token_count

        return mean_embedding.squeeze(0)  # Shape: (d_model)

In [7]:
import time
def train_model(model, dataset, epochs: int=10, batch_size: int=1, lr=2.5e-5, validation_ratio=0.1, device='cuda', collate_fn=None):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    data_size = len(dataset)
    val_size = int(data_size*validation_ratio)
    train_dataset, val_dataset = random_split(dataset, [data_size-val_size, val_size]) 
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    for epoch in range(epochs):
        running_loss = 0
        preds = []
        ground = []
        start_time = time.perf_counter()
        
        model.train()
        for inputs, labels in train_dataloader:            
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            logits = model(inputs)
            loss = loss_fn(logits, labels)
            loss.backward()
            preds.extend(logits.argmax(dim=1).cpu().detach())
            ground.extend(labels.cpu().detach())

            #nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()

            running_loss += loss.item()

        train_accuracy, train_recall, train_precision = compute_metrics(ground, preds)
        
        model.eval()
        preds = []
        ground = []
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                logits = model(inputs)
                preds.extend(logits.argmax(dim=1).cpu().detach())
                ground.extend(labels.cpu().detach())
                val_loss += loss_fn(logits, labels)

        val_accuracy, val_recall, val_precision = compute_metrics(ground, preds)
            
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {running_loss/len(train_dataloader)}, Val Loss: {val_loss/len(val_dataloader)}, Time: {time.perf_counter()-start_time:.2f}")
        print(f"   Train Accuracy: {train_accuracy:.4f}, Train Recall: {train_recall:.4f}, Train Precision: {train_precision:.4f}")
        print(f"   Val Accuracy: {val_accuracy:.4f}, Val Recall: {val_recall:.4f}, Val Precision: {val_precision:.4f}")

def compute_metrics(y_true, y_pred, average: str="micro"):
    """
    Compute per-base accuracy, sensitivity (recall), and PPV (precision).
    Args:
        y_true (list): Ground truth labels
        y_pred (list): Predicted labels
        average (str): Method for multiclass averaging (macro, micro)
    Returns:
        accuracy, recall, precision (all floats)
    """
    y_true = torch.stack(y_true).numpy().flatten().tolist()
    y_pred = torch.stack(y_pred).numpy().flatten().tolist()
    print(y_true)
    print(y_pred)

    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, average=average)
    precision = precision_score(y_true, y_pred, average=average)
    return accuracy, recall, precision


In [8]:
dataset = MicrobiomeDataset(tokenized_files, labels, trim_to = 1000)

In [9]:
model = MicrobiomeLM(d_model=144, n_layers = 4)
device = "cuda" if torch.cuda.is_available() else "cpu"

def custom_collate_fn(batch):
    xs, ys = zip(*batch)  # [(seq_len, feature_dim), label]

    # Pad the sequences along seq_len dimension
    # Resulting shape: (batch_size, max_seq_len, feature_dim)
    padded_xs = pad_sequence(xs, batch_first=True)  

    # Stack the labels
    ys = torch.stack(ys)

    return padded_xs, ys

train_model(model, 
            dataset, 
            batch_size=1,
            validation_ratio=0.1,
            collate_fn=custom_collate_fn, 
            device=device)

[1, 0, 1, 1, 0, 2, 0, 2, 2, 2, 1, 0, 0, 0, 1, 1, 0, 2, 2, 0, 1, 2, 1, 1, 0, 0, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1]
[2, 2, 1]
[0, 0, 0]
Epoch 1/10, Train Loss: 1.1698310595971566, Val Loss: 1.2158734798431396, Time: 139.11
   Train Accuracy: 0.2593, Train Recall: 0.2593, Train Precision: 0.2593
   Val Accuracy: 0.0000, Val Recall: 0.0000, Val Precision: 0.0000
[2, 1, 0, 1, 0, 1, 1, 1, 1, 2, 2, 2, 0, 2, 2, 2, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 2]
[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1]
[2, 1, 2]
[1, 1, 1]
Epoch 2/10, Train Loss: 1.1555619438489277, Val Loss: 1.1253464221954346, Time: 129.80
   Train Accuracy: 0.2593, Train Recall: 0.2593, Train Precision: 0.2593
   Val Accuracy: 0.3333, Val Recall: 0.3333, Val Precision: 0.3333
[0, 1, 1, 0, 0, 0, 1, 1, 2, 1, 0, 0, 2, 2, 2, 0, 2, 1, 0, 1, 1, 0, 2, 1, 2, 0, 2]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[2, 2, 1]
[0, 


KeyboardInterrupt



In [10]:
sum(p.numel() for p in MicrobiomeLM(d_model=144, n_layers = 4).parameters())

3411203